# 04 - Cross-Validation & Data Leakage

Two ideas that decide whether your reported score is real.

1. **Cross-validation** - one train/test split gives you a number with a lot of luck in it.
2. **Data leakage** - the model sees information at training time that it would not have
   in the real world, so the score is optimistic and the model fails in production.

Notebook 01 used `Pipeline`. This notebook shows you *why* it exists.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
np.random.seed(42)

heart = pd.read_csv("../data/heart-disease.csv")
X = heart.drop("target", axis=1)
y = heart["target"]
print(X.shape, y.value_counts().to_dict())

## 1. A single split is a lottery ticket

Same model, same data, same code - only `random_state` on the split changes.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

scores = []
for seed in range(20):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    m = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
    scores.append(m.score(X_te, y_te))

scores = np.array(scores)
print(f"lowest  : {scores.min():.3f}")
print(f"highest : {scores.max():.3f}")
print(f"spread  : {scores.max() - scores.min():.3f}   <- pure luck of the split")
print(f"mean    : {scores.mean():.3f} +/- {scores.std():.3f}")

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.bar(range(20), scores, color="#4C78A8")
plt.axhline(scores.mean(), color="#E45756", ls="--", label=f"mean {scores.mean():.3f}")
plt.ylim(scores.min() - 0.05, scores.max() + 0.03)
plt.xlabel("random_state of the split"); plt.ylabel("test accuracy")
plt.title("The same model scored 20 different ways")
plt.legend()
plt.show()

> If you report the best of those, you are not reporting your model - you are reporting
> your luck. Cross-validation replaces one split with k splits and averages them.

## 2. K-Fold and Stratified K-Fold

In [ ]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold

model = RandomForestClassifier(n_estimators=100, random_state=42)

cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print("fold scores:", np.round(cv_scores, 3))
print(f"mean {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

### Stratify your folds for classification

`StratifiedKFold` keeps the class balance of the full dataset inside every fold.
For classification this is what you want, and `cross_val_score` already uses it by
default when you pass an integer `cv` and a classifier. Being explicit is still good habit.

In [ ]:
plain = cross_val_score(model, X, y, cv=KFold(5, shuffle=True, random_state=42))
strat = cross_val_score(model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42))

print(f"KFold           : {plain.mean():.3f} +/- {plain.std():.3f}")
print(f"StratifiedKFold : {strat.mean():.3f} +/- {strat.std():.3f}")

### Choosing a CV splitter

| Splitter | Use when |
|---|---|
| `KFold` | plain regression, rows are independent |
| `StratifiedKFold` | **classification** - the default you want |
| `GroupKFold` | rows are grouped (several visits per patient, several photos per user) and a group must not straddle the split |
| `TimeSeriesSplit` | the data has a time order - never train on the future |

Getting this wrong is itself a form of leakage. If the same patient appears in both train
and test, the model can recognise the patient rather than learn the disease.

## 3. More than one metric at a time

In [ ]:
from sklearn.model_selection import cross_validate

results = cross_validate(
    model, X, y, cv=5,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
    return_train_score=True)

summary = pd.DataFrame({
    k.replace("test_", ""): [v.mean(), v.std()]
    for k, v in results.items() if k.startswith("test_")
}, index=["mean", "std"]).T.round(3)
summary

## 4. Data leakage - the expensive mistake

Here is the wrong way to scale, which almost every beginner writes at least once:

```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)          # <- fitted on ALL the data
X_train, X_test = train_test_split(X_scaled)
```

The scaler computed the mean and standard deviation using the test rows. Information from
the test set has leaked into training. The reported score goes up; real-world performance
does not.

Let us measure the size of the lie. To make it visible, we use a feature-selection step,
where leakage bites hardest.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Random noise features - there is genuinely NOTHING to learn here.
rng = np.random.RandomState(0)
n_samples = 200
X_noise = pd.DataFrame(rng.randn(n_samples, 2000))
y_noise = pd.Series(rng.randint(0, 2, n_samples))

# --- WRONG: select features using the whole dataset, then cross-validate ---
selector = SelectKBest(f_classif, k=20).fit(X_noise, y_noise)
X_selected = selector.transform(X_noise)
leaky = cross_val_score(LogisticRegression(max_iter=1000), X_selected, y_noise, cv=5)

# --- RIGHT: selection happens inside the pipeline, refit on each fold ---
honest_pipe = Pipeline([
    ("select", SelectKBest(f_classif, k=20)),
    ("clf", LogisticRegression(max_iter=1000)),
])
honest = cross_val_score(honest_pipe, X_noise, y_noise, cv=5)

print(f"LEAKY  score on pure noise : {leaky.mean():.3f}")
print(f"HONEST score on pure noise : {honest.mean():.3f}")
print("\nTruth: the data is random. Anything above ~0.50 is an illusion.")

That is the whole lesson. The leaky version invents a model that looks useful and is
worth nothing. The pipeline version correctly reports "there is no signal here".

### The rule

> **Every step that learns something from the data - imputers, scalers, encoders,
> feature selectors, PCA - must be fitted inside the cross-validation loop.**
> A `Pipeline` guarantees that for free.

In [ ]:
# The correct pattern, end to end.
correct = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

cv_scores = cross_val_score(correct, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42))
print(f"honest cross-validated accuracy: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

## 5. Other leakage traps to recognise

- **A target-derived feature.** Predicting churn using `cancellation_date`. The column is
  only filled in *after* the event you are predicting.
- **Duplicate rows** split across train and test.
- **Time travel.** Using next week's data to predict this week. Always `TimeSeriesSplit`.
- **Preprocessing before splitting.** The case demonstrated above.

Sanity check: if your accuracy jumps to 0.99 on a hard problem, assume leakage and go
looking for it before you celebrate.

---
## Exercises

1. Re-run the 20-seed experiment with `LogisticRegression` inside a scaling pipeline.
   Is the spread bigger or smaller than the Random Forest's? Why might that be?
2. Change `cv=5` to `cv=10` in `cross_val_score`. What happens to the mean, and what
   happens to the standard deviation? What is the cost?
3. Build the leaky-vs-honest comparison again with `k=100` selected noise features.
   Does the illusion get stronger or weaker?
4. Write a pipeline for `car-sales-extended-missing-data.csv` that imputes, encodes and
   scales, then report a 5-fold cross-validated R^2. Confirm no step touches the test fold.
5. **Stretch:** invent a leaky feature for `heart-disease.csv` - for example
   `target + small noise` - add it as a column and watch cross-validated accuracy hit
   ~1.00. Then explain how you would have caught it in a code review.